In [31]:
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.tensorboard import SummaryWriter
import torchmetrics

import matplotlib.pyplot as plt

# Definition of parameters

In [42]:
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print("Using device:", device)
learning_rate = 0.00001
batch_size = 50
experiment_name = 'prova1'

Using device: mps


# Data loading

In [43]:
class Dataset(torch.utils.data.Dataset):

    def __init__(self, csv):
        # read the csv file
        self.df = pd.read_csv(csv, sep=',')
        # self.df = self.df.dropna(axis=0)
        # save cols
        self.output_cols = ['Cover_Type']
        self.input_cols = list(set(self.df.columns)-set(self.output_cols))


    def __len__(self):
        # here i will return the number of samples in the dataset
        return len(self.df)


    def __getitem__(self, idx):
        # here i will load the file in position idx
        cur_sample = self.df.iloc[idx]
        # split in input / ground-truth
        cur_sample_x = cur_sample[self.input_cols]
        cur_sample_y = cur_sample[self.output_cols]
        # convert to torch format
        cur_sample_x = torch.tensor(cur_sample_x.tolist())
        cur_sample_y = torch.tensor(cur_sample_y.tolist())
        # remove dimension on gt
        cur_sample_y = cur_sample_y.squeeze()
        # convert to int
        cur_sample_y = cur_sample_y.long() - 1
        # return values
        return cur_sample_x, cur_sample_y


In [44]:
# create train and validation datasets
train_ds = Dataset('../dataset/ForestDataset/train.csv')
val_ds =  Dataset("../dataset/ForestDataset/val.csv")

In [45]:
# create train dataloader
train_dl = torch.utils.data.DataLoader(
    train_ds,
    batch_size = batch_size,
    drop_last = True,
    shuffle = True,
    num_workers = 0
)
# create validation dataloader
val_dl = torch.utils.data.DataLoader(
    val_ds,
    batch_size = batch_size,
    drop_last = False,
    shuffle = False,
    num_workers = 0
)

# Network definition

In [46]:
# define network

class Net(nn.Module):

    def __init__(self, n_inputs, n_classes):
        # initialize super class
        super(Net, self).__init__()
        self.layer1 = nn.Linear(n_inputs,64)
        self.layer2 = nn.ReLU()
        self.layer3 = nn.Linear(64,32)
        self.layer4 = nn.ReLU()
        self.layer5 = nn.Linear(32,16)
        self.layer6 = nn.ReLU()
        self.layer7 = nn.Linear(16, n_classes)


    def forward(self, x):
        # apply layers in cascade
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        x = self.layer5(x)
        x = self.layer6(x)
        x = self.layer7(x)
        # return output
        return x


# Define validation routine

In [47]:
# create validation routine
def validate(net, dl, n_classes, device):
    # create metric objects
    tm_acc = torchmetrics.Accuracy(task='multiclass', num_classes=n_classes, average= 'macro', top_k=1)
    tm_con = torchmetrics.ConfusionMatrix(task="multiclass", num_classes=n_classes)
    # move metric to device
    tm_acc.to(device)
    tm_con.to(device)
    # set network in eval mode
    net.eval()
    # at the end of epoch, validate model
    for inp, gt in dl:
        # move batch to gpu
        inp = inp.to(device)
        gt = gt.to(device)
        # remove singleton dimension
        gt = gt.squeeze()
        # get output
        with torch.no_grad():
            # perform prediction
            logits = net(inp)
        # update metrics
        tm_acc.update(logits, gt)
        tm_con.update(logits, gt)

    # at the end, compute metric
    acc = tm_acc.compute()
    con = tm_con.compute()
    # set network in training mode
    net.train()
    # return score
    return acc, con
    
    
    
        

In [48]:
# # try validation function before training
# acc, conf = validate(net, val_dl, n_classes, device)
# # print
# print(f'{acc:0.4f}')

# Train

In [49]:
import shutil
# %load_ext tensorboard
%reload_ext tensorboard
%tensorboard --logdir={experiment_name}

ERROR: Failed to launch TensorBoard (exited with 1).
Contents of stderr:
Traceback (most recent call last):
  File "/Library/Frameworks/Python.framework/Versions/3.14/bin/tensorboard", line 3, in <module>
    from tensorboard.main import run_main
  File "/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/tensorboard/main.py", line 27, in <module>
    from tensorboard import default
  File "/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/tensorboard/default.py", line 30, in <module>
    import pkg_resources
ModuleNotFoundError: No module named 'pkg_resources'

In [50]:
# initialize the network
net = Net(n_inputs, n_classes)

# move to device
net = net.to(device)

# define optimizer
optimizer = torch.optim.Adam(params=net.parameters(), lr=learning_rate)

# define summary writer
writer = SummaryWriter(experiment_name)

# initialize iteration number
n_iter = 0

# define best validation value
best_acc = None

# define loss function
loss_fun = nn.CrossEntropyLoss()

# for each epoch
for cur_epoch in range(100):
    # plot current epoch
    writer.add_scalar("epoch", cur_epoch, n_iter)
    # for each batch
    for inp, gt in train_dl:
        # move batch to gpu
        inp = inp.to(device)
        gt = gt.to(device)
        # reset gradients
        optimizer.zero_grad()
        # get output
        logits = net(inp)
        # compute loss
        loss = loss_fun(logits, gt.squeeze())
        # compute backward
        loss.backward()
        # update weights
        optimizer.step()
        # plot
        writer.add_scalar("loss", loss.item(), n_iter)
        n_iter += 1
        
    # at the end, validate model
    cur_acc, cur_conf_mat = validate(net, val_dl, n_classes, device)
    # plot validation
    writer.add_scalar("accuracy", cur_acc.item(), n_iter)
    # check if it is the best model so far
    if best_acc is None or cur_acc > best_acc:
        # define new best val
        best_acc = cur_acc
        # save current model as best
        torch.save({
            'net': net.state_dict(),
            'opt': optimizer.state_dict(),
            'epoch': cur_epoch
        }, experiment_name + '_best.pth')
    # save last model
    torch.save({
        'net': net.state_dict(),
        'opt': optimizer.state_dict(),
        'epoch': cur_epoch
    }, experiment_name + '_last.pth')
    

# Test

In [51]:
# create test dataset
test_ds =  Dataset('../dataset/ForestDataset/test.csv')

# create dataloader
test_dl = torch.utils.data.DataLoader(
    test_ds,
    batch_size = batch_size,
    drop_last = False,
    shuffle = False,
    num_workers = 0
)

In [52]:
# load best network
state = torch.load(experiment_name + '_best.pth')
net.load_state_dict(state['net'])

<All keys matched successfully>

In [53]:
test_accuracy, test_conf_mat = validate(net, test_dl, n_classes, device)

In [54]:
print(f'The model scored an accuracy of {test_accuracy:0.04f} over the testset.')

The model scored an accuracy of 0.6313 over the testset.


In [57]:
import seaborn as sns

def plot_confusion_matrix(conf_mat):
    cm = sns.light_palette("blue", as_cmap=True)
    x=pd.DataFrame(conf_mat.cpu())
    x=x.style.background_gradient(cmap=cm)
    display(x)

In [58]:
plot_confusion_matrix(test_conf_mat)

,0,1,2,3,4,5,6
0,119,42,1,0,19,2,18
1,63,108,2,0,32,4,4
2,0,5,100,40,12,90,0
3,0,0,12,155,0,13,0
4,20,40,18,0,160,6,0
5,1,9,45,22,15,128,0
6,34,1,0,0,3,0,169
